# Laboratorio · Sesión 4 — Automatización: funciones, pipelines y reportes
**PY-EAN · Python para Procesamiento, Calidad y Automatización de Datos** · Universidad Ean

**Caso:** cada mes llega un archivo de aportes por planilla (enero a septiembre de 2026 = 9 archivos).
Hoy alguien los copia y pega en Excel, revisa a ojo y arma el reporte de recaudo para la dirección.
Tu misión: **convertir ese trabajo manual en un pipeline** que lea los 9 archivos, los valide, marque alertas
y genere solo un **reporte Excel** con gráfico.

**Cómo trabajar:** Shift + Enter, de arriba hacia abajo. Donde veas `# ✏️ TU CÓDIGO AQUÍ`, escribe tu solución.
Si Colab se reinicia: **Entorno de ejecución → Ejecutar todas**.

> Todos los datos son sintéticos. No corresponden a personas reales.

## Parte 0 · Preparar el entorno (5 min)

In [1]:
# ▶️ Ejecuta esta celda primero (Shift + Enter)
import os
import pandas as pd

URL_DATOS = "https://raw.githubusercontent.com/Hackwy402/python-procesamiento-ean/main/datos/"
# Si subiste la carpeta "datos" a Colab (o trabajas en tu PC) se usa la copia local;
# si no, se leen los datos directamente desde GitHub.
for carpeta in ["datos/", "../datos/", "/content/datos/"]:
    if os.path.exists(carpeta + "afiliados.csv"):
        RUTA = carpeta
        break
else:
    RUTA = URL_DATOS

pd.set_option("display.max_columns", 30)
print("Leyendo datos desde:", RUTA)

from datetime import datetime
import matplotlib.pyplot as plt

SMMLV = 1_750_905        # salario mínimo 2026 (dato de referencia del curso)
TASA = 0.16              # aporte a pensión = 16 % del IBC
COLUMNAS = ["id_afiliado", "periodo", "ibc", "valor_aporte", "fecha_pago", "operador"]
SALIDA = "salidas"
os.makedirs(SALIDA, exist_ok=True)

Leyendo datos desde: ../datos/


## Parte 1 · De celdas sueltas a funciones (10 min)
Así se vería leer **un** mes “a mano”:

In [10]:
enero = pd.read_csv(RUTA + "aportes/aportes_2026-01.csv")
print(enero.shape)
enero.head()

(1770, 6)


,id_afiliado,periodo,ibc,valor_aporte,fecha_pago,operador
0,AF100026,2026-01,1819000,291040,2026-02-02,Enlace Operativo
1,AF100492,2026-01,1760000,281600,2026-05-02,Enlace Operativo
2,AF101579,2026-01,8985000,1437600,2026-02-20,Aportes en Línea
3,AF100510,2026-01,5522000,883520,2026-02-15,Mi Planilla
4,AF100689,2026-01,4166000,666560,2026-02-15,Aportes en Línea


**✏️ Ejercicio 1.** Escribe la función `ruta_mes(mes)` que reciba un número de mes (1 a 12) y devuelva la ruta del
archivo, p. ej. `ruta_mes(3)` → `RUTA + "aportes/aportes_2026-03.csv"`.
Pista: `f"{mes:02d}"` rellena con cero a la izquierda.

In [24]:
def ruta_mes(mes):
    return RUTA + f"aportes/aportes_2026-{mes:02d}.csv"

print(ruta_mes(5))


../datos/aportes/aportes_2026-05.csv


**✏️ Ejercicio 2.** Escribe `marcar_aporte_incorrecto(df)` que agregue la columna booleana `aporte_incorrecto`:
`True` si `valor_aporte` se aleja más de **1 peso** de `ibc * TASA` (redondeado). Pruébala con `enero` y cuenta
cuántos aportes incorrectos hay en enero.

In [27]:
enero[enero["aporte_incorrecto"]].head(17)

,id_afiliado,periodo,ibc,valor_aporte,fecha_pago,operador,aporte_incorrecto
19,AF100377,2026-01,1751000,188856,2026-02-12,Mi Planilla,True
457,AF102556,2026-01,5527000,545199,2026-02-17,Enlace Operativo,True
622,AF100186,2026-01,4686000,511030,2026-02-15,Asopagos,True
713,AF102138,2026-01,2305000,243920,2026-02-08,SOI,True
792,AF101607,2026-01,1751000,203334,2026-02-09,Enlace Operativo,True
822,AF102391,2026-01,1751000,165985,2026-02-19,SOI,True
902,AF100934,2026-01,5617000,655240,2026-02-12,Asopagos,True
908,AF100067,2026-01,2761000,392203,2026-02-05,SOI,True
942,AF100014,2026-01,1751000,230649,2026-02-15,Enlace Operativo,True
976,AF101900,2026-01,3967000,472496,2026-02-05,Mi Planilla,True


In [25]:
def marcar_aporte_incorrecto(df, tasa=TASA, tolerancia=1):
    """Agrega 'aporte_incorrecto': True si valor_aporte != ibc * tasa (± tolerancia)."""
    esperado = df["ibc"] * tasa
    df = df.copy()  # no modificar el original
    df["aporte_incorrecto"] = (df["valor_aporte"] - esperado).abs().round() > tolerancia
    return df

enero = marcar_aporte_incorrecto(enero)
print("Aportes incorrectos en enero:", enero["aporte_incorrecto"].sum())


Aportes incorrectos en enero: 17


## Parte 2 · Muchos archivos… y uno que llega distinto (15 min)
**✏️ Ejercicio 3.** Crea la lista `rutas` con los 9 archivos (meses 1 a 9) usando `ruta_mes` y una
*list comprehension*. Imprime cuántas rutas tienes.

> 💡 En tu PC podrías usar `glob.glob("datos/aportes/aportes_*.csv")`, pero glob **no funciona con URLs**:
> por eso construimos la lista nosotros.

In [28]:
rutas = [ruta_mes(m) for m in range(1, 10)]
print("Rutas:", len(rutas))
rutas[:2]


Rutas: 9


['../datos/aportes/aportes_2026-01.csv',
 '../datos/aportes/aportes_2026-02.csv']

**✏️ Ejercicio 4.** Recorre `rutas` con un `for`, lee cada archivo con `pd.read_csv` (sin opciones) e imprime el nombre
del archivo y su `shape`. **¿Qué mes se ve raro? ¿Por qué?**

In [15]:
for ruta in rutas:
    df = pd.read_csv(ruta)
    print(ruta.split("/")[-1], df.shape)

aportes_2026-01.csv (1770, 6)
aportes_2026-02.csv (1757, 6)
aportes_2026-03.csv (1770, 6)
aportes_2026-04.csv (1775, 6)
aportes_2026-05.csv (1761, 6)
aportes_2026-06.csv (1764, 6)
aportes_2026-07.csv (1785, 1)
aportes_2026-08.csv (1749, 6)
aportes_2026-09.csv (1745, 6)


In [33]:
julio = pd.read_csv(ruta_mes(7))
print(julio.columns.tolist())

['ID_AFILIADO;PERIODO;IBC;VALOR_APORTE;FECHA_PAGO;OPERADOR']


In [20]:
def validar_estructura(df, esperadas=COLUMNAS):
    """Devuelve las columnas esperadas que NO están en df."""
    return [c for c in esperadas if c not in df.columns]

print("Enero, faltan:", validar_estructura(enero))
print("Julio, faltan:", validar_estructura(julio))


Enero, faltan: []
Julio, faltan: ['id_afiliado', 'periodo', 'ibc', 'valor_aporte', 'fecha_pago', 'operador']


✍️ *¿Qué pasó con julio?* …

In [34]:
# Miremos julio de cerca
julio = pd.read_csv(ruta_mes(7))
print(julio.columns.tolist())

['ID_AFILIADO;PERIODO;IBC;VALOR_APORTE;FECHA_PAGO;OPERADOR']


**✏️ Ejercicio 5.** Escribe `validar_estructura(df, esperadas=COLUMNAS)` que devuelva la **lista de columnas que faltan**
(lista vacía si está todo bien). Pruébala con `enero` y con `julio`.

In [35]:
def validar_estructura(df, esperadas=COLUMNAS):
    """Devuelve las columnas esperadas que NO están en df."""
    # ✏️ TU CÓDIGO AQUÍ
    pass

print("Enero, faltan:", validar_estructura(enero))
print("Julio, faltan:", validar_estructura(julio))

Enero, faltan: None
Julio, faltan: None


**✏️ Ejercicio 6.** Escribe `leer_aportes(ruta)` **robusta**: que detecte el separador (`sep=None, engine="python"`)
y normalice los nombres de columnas (`.str.strip().str.lower()`). Comprueba que ahora julio pasa la validación.

In [37]:
def leer_aportes(ruta):
    """Lee un archivo de aportes detectando el separador y normalizando columnas."""
    df = pd.read_csv(ruta, sep=None, engine="python")
    df.columns = df.columns.str.strip().str.lower()
    return df

julio = leer_aportes(ruta_mes(7))
print(julio.shape, "| faltan:", validar_estructura(julio))

(1785, 6) | faltan: None


**✏️ Ejercicio 7.** Completa `cargar_aportes(meses)` con **manejo de errores y bitácora**: para cada mes intenta
leer y validar; si faltan columnas lanza `ValueError`; si todo sale bien agrega la columna `archivo_origen` y guarda el
DataFrame en `partes`. Registra cada evento con `registrar(...)`. Al final concatena con `pd.concat`.

Para probar que el `except` funciona, la celda incluye el mes **10** (que no existe).

In [38]:
bitacora = []

def registrar(archivo, evento, filas=None):
    """Agrega una línea a la bitácora con la hora actual."""
    bitacora.append({"hora": datetime.now().strftime("%H:%M:%S"),
                     "archivo": archivo, "evento": evento, "filas": filas})

In [40]:
bitacora = []

def registrar(archivo, evento, filas=None):
    """Agrega una línea a la bitácora con la hora actual."""
    bitacora.append({"hora": datetime.now().strftime("%H:%M:%S"),
                     "archivo": archivo, "evento": evento, "filas": filas})

In [41]:
def cargar_aportes(meses=range(1, 10)):
    """Lee, valida y concatena los archivos de aportes de los meses indicados."""
    partes = []
    for m in meses:
        ruta = ruta_mes(m)
        nombre = ruta.split("/")[-1]
        try:
            df = leer_aportes(ruta)
            faltan = validar_estructura(df)
            if faltan:
                raise ValueError(f"faltan columnas {faltan}")
            df["archivo_origen"] = nombre
            partes.append(df)
            registrar(nombre, "OK", len(df))
        except Exception as e:
            registrar(nombre, f"ERROR: {type(e).__name__}: {e}"[:120])
    return pd.concat(partes, ignore_index=True)

aportes = cargar_aportes(range(1, 11))
print(aportes.shape)
pd.DataFrame(bitacora)

(15876, 7)


,hora,archivo,evento,filas
0,09:55:56,aportes_2026-01.csv,OK,1770.0
1,09:55:56,aportes_2026-02.csv,OK,1757.0
2,09:55:56,aportes_2026-03.csv,OK,1770.0
3,09:55:56,aportes_2026-04.csv,OK,1775.0
4,09:55:56,aportes_2026-05.csv,OK,1761.0
5,09:55:56,aportes_2026-06.csv,OK,1764.0
6,09:55:56,aportes_2026-07.csv,OK,1785.0
7,09:55:56,aportes_2026-08.csv,OK,1749.0
8,09:55:56,aportes_2026-09.csv,OK,1745.0
9,09:55:56,aportes_2026-10.csv,ERROR: FileNotFoundError: [Errno 2] No such fi...,NaN


## Parte 3 · Limpiar y marcar alertas (15 min)
**✏️ Ejercicio 8.** Escribe `limpiar_aportes(df)` que:
1. elimine duplicados exactos usando solo las `COLUMNAS` del negocio (`subset=COLUMNAS`);
2. convierta `fecha_pago` a fecha (`pd.to_datetime(..., errors="coerce")`).

¿Cuántas filas duplicadas se eliminaron?

In [ ]:
def limpiar_aportes(df):
    """Quita duplicados y corrige tipos."""
    # ✏️ TU CÓDIGO AQUÍ
    return df

filas_antes = len(aportes)
aportes = limpiar_aportes(aportes)
print("Duplicados eliminados:", filas_antes - len(aportes), "| filas:", len(aportes))

**✏️ Ejercicio 9.** Escribe `marcar_alertas(df)` que agregue tres columnas booleanas y una de resumen:
- `aporte_incorrecto` → reutiliza `marcar_aporte_incorrecto` (¡para eso escribimos funciones!);
- `pago_tardio` → `fecha_pago` posterior al **último día del mes siguiente** al periodo
  (pista: `pd.to_datetime(df["periodo"] + "-01") + pd.offsets.MonthEnd(2)`);
- `ibc_bajo_minimo` → `ibc < SMMLV`;
- `tiene_alerta` → `True` si cualquiera de las tres es `True`.

In [ ]:
def marcar_alertas(df):
    """Marca aportes incorrectos, pagos tardíos e IBC por debajo del mínimo."""
    # ✏️ TU CÓDIGO AQUÍ
    return df

aportes = marcar_alertas(aportes)
aportes[["aporte_incorrecto", "pago_tardio", "ibc_bajo_minimo", "tiene_alerta"]].sum()

## Parte 4 · Resumir y graficar (10 min)
**✏️ Ejercicio 10.** Escribe `resumir(df)` que devuelva una tabla **por periodo** con: `aportantes` (únicos),
`recaudo` (suma de `valor_aporte`), `aportes_incorrectos`, `pagos_tardios` e `ibc_bajo_minimo`.

In [ ]:
def resumir(df):
    """Resumen mensual para la dirección."""
    # ✏️ TU CÓDIGO AQUÍ  (groupby("periodo").agg(aportantes=("id_afiliado", "nunique"), ...))
    pass

resumen = resumir(aportes)
resumen

**✏️ Ejercicio 11.** Resume **por operador**: número de aportes, recaudo total y % de pagos tardíos. ¿Qué operador tiene más recaudo?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

**✏️ Ejercicio 12.** Escribe `graficar_recaudo(resumen, ruta_png)` que haga un gráfico de barras del recaudo mensual (en millones) y lo **guarde** como PNG.

In [ ]:
def graficar_recaudo(resumen, ruta_png):
    """Gráfico de barras del recaudo mensual, guardado como PNG."""
    fig, ax = plt.subplots(figsize=(9, 4))
    # ✏️ TU CÓDIGO AQUÍ  (ax.bar(...), título, etiqueta del eje Y)
    fig.tight_layout()
    fig.savefig(ruta_png, dpi=120)
    plt.show()
    return ruta_png

ruta_png = graficar_recaudo(resumen, f"{SALIDA}/recaudo_mensual.png")

## Parte 5 · Reporte Excel automático (15 min)
**✏️ Ejercicio 13.** Escribe `exportar_reporte(...)` que cree un Excel **nombrado con la fecha de hoy**
(`reporte_aportes_AAAA-MM-DD.xlsx`) con las hojas: `resumen_mensual`, `por_operador`, `periodo_x_operador`
(tabla dinámica de recaudo), `alertas` (solo filas con `tiene_alerta`) y `bitacora`.

In [ ]:
def exportar_reporte(aportes, resumen, por_operador, carpeta=SALIDA):
    """Escribe el reporte Excel con varias hojas y devuelve su ruta."""
    os.makedirs(carpeta, exist_ok=True)
    ruta_xlsx = f"{carpeta}/reporte_aportes_{datetime.now():%Y-%m-%d}.xlsx"
    matriz = aportes.pivot_table(index="periodo", columns="operador",
                                 values="valor_aporte", aggfunc="sum")
    alertas = aportes[aportes["tiene_alerta"]].copy()
    alertas["fecha_pago"] = alertas["fecha_pago"].dt.date
    with pd.ExcelWriter(ruta_xlsx, engine="openpyxl") as writer:
        resumen.to_excel(writer, sheet_name="resumen_mensual", index=False)
        # ✏️ TU CÓDIGO AQUÍ: las otras 4 hojas
    return ruta_xlsx

ruta_xlsx = exportar_reporte(aportes, resumen, por_operador)
print("Reporte:", ruta_xlsx)

**✏️ Ejercicio 14.** Completa `dar_formato(ruta_xlsx, ruta_png)` para que **cada hoja** tenga: encabezado en negrita
blanca con fondo azul, ancho de columnas ajustado, formato de miles `#,##0` en los números y encabezado fijo.
Además, inserta el gráfico en la hoja `resumen_mensual`.

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.drawing.image import Image as XLImage

In [ ]:
def dar_formato(ruta_xlsx, ruta_png=None):
    """Aplica formato básico a todas las hojas e inserta el gráfico."""
    wb = load_workbook(ruta_xlsx)
    for ws in wb.worksheets:
        # ✏️ 1) encabezados (ws[1]): Font(bold=True, color="FFFFFF") y PatternFill("solid", fgColor="1F3864")
        # ✏️ 2) for col in ws.columns: ancho de columna y number_format = "#,##0" en números
        ws.freeze_panes = "A2"
    if ruta_png:
        wb["resumen_mensual"].add_image(XLImage(ruta_png), "H2")
    wb.save(ruta_xlsx)
    return ruta_xlsx

dar_formato(ruta_xlsx, ruta_png)
print("Formato aplicado ✔")

**✏️ Ejercicio 15.** Descarga el reporte a tu computador y ábrelo en Excel. (En tu PC, fuera de Colab, la celda solo muestra la ruta.)

In [ ]:
try:
    from google.colab import files
    files.download(ruta_xlsx)
except ImportError:
    print("No estás en Colab. El reporte quedó en:", os.path.abspath(ruta_xlsx))

## Parte 6 · El pipeline completo (10 min)
**✏️ Ejercicio 16.** Une todo en `ejecutar_pipeline(meses, carpeta_salida)`: limpiar la bitácora → cargar → limpiar →
marcar alertas → resumir → graficar → exportar → dar formato. Debe devolver la ruta del Excel.
Ejecútala para **todos los meses** y luego **solo para el primer trimestre** (meses 1 a 3).

In [ ]:
def ejecutar_pipeline(meses=range(1, 10), carpeta_salida=SALIDA):
    """Cargar → validar → limpiar → alertas → resumir → exportar."""
    bitacora.clear()
    # ✏️ TU CÓDIGO AQUÍ — llama, en orden, a las funciones que ya escribiste
    pass

ejecutar_pipeline()

In [ ]:
# ✏️ Ejecuta el pipeline solo para el primer trimestre en la carpeta f"{SALIDA}/trimestre_1"

## 🎯 Reto de cierre — Aportes huérfanos
Un aporte es **huérfano** si su `id_afiliado` **no existe** en `afiliados.csv` (integridad referencial, Sesión 3).

1. Lee `afiliados.csv` y marca en `aportes` la columna `huerfano`.
2. ¿Cuántos aportes huérfanos hay y cuánto dinero suman?
3. **Bonus:** agrégalos como una hoja `huerfanos` del reporte (modifica `exportar_reporte` o abre el archivo con
   `pd.ExcelWriter(ruta_xlsx, mode="a", engine="openpyxl")`).

In [ ]:
# ✏️ TU CÓDIGO AQUÍ

---
### 🚀 ¿Y ahora? Llevarlo a producción
- **Archivo → Descargar → Descargar .py** convierte este notebook en script.
- En el repositorio tienes `pipeline_aportes.py`, la versión “de producción” de este laboratorio:
  `python pipeline_aportes.py --datos ../datos --salida salidas --meses 1-9`
- Prográmalo con el **Programador de tareas de Windows** o `cron` para que corra solo cada mes.